# 🔨 Crack 3D Reconstruction + Repair
Run cells 1–7 in order. **GPU runtime recommended.**

**Three outputs per run:**
- 🪨 Cracked mesh — original surface with crack depression
- 🔧 Filled mesh — crack levelled flush using inpainted depth + texture
- 📊 8-panel analysis report — depth maps, severity, repaired previews

In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────────────────────
# Run once. Restart runtime if prompted, then re-run from Cell 2 onward.
%pip install -q gradio open3d transformers pillow accelerate opencv-python matplotlib trimesh pygltflib


In [ ]:
# ── Cell 2: Imports & model ───────────────────────────────────────────────────
import os, tempfile, traceback
import cv2
import numpy as np
from PIL import Image
from scipy.spatial import cKDTree

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

import torch
from transformers import pipeline
import open3d as o3d
import trimesh
import gradio as gr

if torch.cuda.is_available():
    device = 0
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    device = -1
    print("⚠️  CPU only — inference will be slow")

print("Loading Depth-Anything-V2-Large (~1.3 GB, cached after first run)…")
depth_pipe = pipeline(
    "depth-estimation",
    model="depth-anything/Depth-Anything-V2-Large-hf",
    device=device,
)
print("✅ Model ready.")


In [ ]:
# ── Cell 3: Helpers ───────────────────────────────────────────────────────────

def classify_severity(max_disp: float, crack_area_pct: float):
    """Return (label, hex_color).  max_disp is on the 0-1000 normalised scale."""
    score = (max_disp / 1000.0) * 0.6 + (crack_area_pct / 100.0) * 0.4
    if score < 0.05:  return "Hairline", "#4CAF50"
    if score < 0.15:  return "Minor",    "#8BC34A"
    if score < 0.30:  return "Moderate", "#FFC107"
    if score < 0.50:  return "Severe",   "#FF5722"
    return            "Critical",        "#B71C1C"


def fig_to_numpy(fig) -> np.ndarray:
    """matplotlib figure → HxWx3 uint8 RGB.  Works on matplotlib >= 3.8."""
    fig.canvas.draw()
    buf = fig.canvas.buffer_rgba()
    w, h = fig.canvas.get_width_height()
    arr = np.frombuffer(buf, dtype=np.uint8).reshape(h, w, 4)
    return arr[:, :, :3].copy()


def build_report(original_rgb, base_depth, crack_mask_raw,
                 displacement_map, final_depth, filled_depth_norm,
                 filled_img_rgb, crack_intensity) -> np.ndarray:
    """
    8-panel analysis report (2 rows x 4 cols):
      [0,0] Original image
      [0,1] AI base depth map        (inferno)
      [0,2] Crack binary mask        (gray)
      [0,3] Displacement heatmap     (jet)
      [1,0] Depth difference Δ       (plasma)
      [1,1] Severity overlay
      [1,2] Inpainted / filled image (how the repaired surface looks in 2D)
      [1,3] Filled depth map         (inferno — levelled crack region)
    """
    crack_px       = (crack_mask_raw > 10).astype(np.uint8)
    crack_area_pct = float(crack_px.mean() * 100.0)
    max_disp       = float(displacement_map.max())
    mean_disp      = (float(displacement_map[crack_px == 1].mean())
                      if crack_px.any() else 0.0)
    depth_diff     = (final_depth - base_depth).astype(np.float32)
    sev_label, _   = classify_severity(max_disp, crack_area_pct)

    sev_cmap  = LinearSegmentedColormap.from_list(
        "sev", ["#4CAF50", "#FFC107", "#FF5722", "#B71C1C"])
    disp_norm = displacement_map / (max_disp + 1e-6)
    sev_rgb   = (sev_cmap(disp_norm)[:, :, :3] * 255).astype(np.uint8)
    alpha     = (crack_px[:, :, None] * 0.80).astype(np.float32)
    bg        = (original_rgb * 0.30).astype(np.uint8)
    sev_over  = np.clip(sev_rgb * alpha + bg * (1 - alpha), 0, 255).astype(np.uint8)

    fig, axes = plt.subplots(2, 4, figsize=(24, 12),
                             facecolor="#1a1a1a",
                             gridspec_kw={"hspace": 0.32, "wspace": 0.08})
    fig.patch.set_facecolor("#1a1a1a")

    panels = [
        (original_rgb,       "Original Image",           None,       False),
        (base_depth,         "AI Base Depth Map",         "inferno",  True),
        (crack_mask_raw,     "Crack Binary Mask",         "gray",     True),
        (displacement_map,   "Displacement Heatmap",      "jet",      True),
        (depth_diff,         "Depth Difference Δ",        "plasma",   True),
        (sev_over,           "Severity Classification",   None,       False),
        (filled_img_rgb,     "Repaired Surface (2D)",     None,       False),
        (filled_depth_norm,  "Levelled Depth Map",        "inferno",  True),
    ]

    for ax, (data, title, cmap, cbar) in zip(axes.flat, panels):
        ax.set_facecolor("#1a1a1a")
        im = ax.imshow(data, cmap=cmap, interpolation="bilinear")
        if cbar:
            cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cb.ax.yaxis.set_tick_params(color="#aaa", labelsize=7)
            plt.setp(cb.ax.yaxis.get_ticklabels(), color="#aaa")
            cb.outline.set_edgecolor("#555")
        ax.set_title(title, color="#eee", fontsize=11, fontweight="bold", pad=6)
        ax.axis("off")

    # Stats + legend under severity panel
    ax_sev = axes[1, 1]
    ax_sev.text(
        0.5, -0.08,
        (f"Severity: {sev_label}   Max Δ: {max_disp:.1f}   "
         f"Mean Δ: {mean_disp:.1f}   Crack area: {crack_area_pct:.2f}%"),
        transform=ax_sev.transAxes,
        ha="center", va="top", fontsize=8, color="#ddd",
        bbox=dict(boxstyle="round,pad=0.4",
                  facecolor="#2a2a2a", edgecolor="#555", linewidth=0.8),
    )
    patches = [
        mpatches.Patch(color="#4CAF50", label="Hairline"),
        mpatches.Patch(color="#8BC34A", label="Minor"),
        mpatches.Patch(color="#FFC107", label="Moderate"),
        mpatches.Patch(color="#FF5722", label="Severe"),
        mpatches.Patch(color="#B71C1C", label="Critical"),
    ]
    ax_sev.legend(handles=patches, loc="upper right", fontsize=7,
                  framealpha=0.55, facecolor="#1a1a1a",
                  edgecolor="#555", labelcolor="#eee")

    fig.suptitle(
        f"Crack Analysis  ·  Displacement strength: {crack_intensity}"
        f"  ·  Overall severity: {sev_label}",
        color="#fff", fontsize=14, fontweight="bold", y=0.99)

    arr = fig_to_numpy(fig)
    plt.close(fig)
    return arr


print("✅ Helpers ready.")


In [ ]:
# ── Cell 4: GLB export helper ─────────────────────────────────────────────────

def _to_glb_with_colors(mesh_o3d, out_path: str) -> str:
    """Export Open3D TriangleMesh with vertex colors to GLB via trimesh."""
    verts     = np.asarray(mesh_o3d.vertices)
    tris      = np.asarray(mesh_o3d.triangles)
    colors_f  = np.asarray(mesh_o3d.vertex_colors)
    colors_u8 = (np.clip(colors_f, 0, 1) * 255).astype(np.uint8)
    rgba      = np.hstack([colors_u8,
                           np.full((len(colors_u8), 1), 255, dtype=np.uint8)])
    tm = trimesh.Trimesh(vertices=verts, faces=tris,
                         vertex_colors=rgba, process=False)
    tm.export(out_path)
    return out_path


In [ ]:
# ── Cell 5: Crack-fill helpers ────────────────────────────────────────────────

def inpaint_depth(depth_f32: np.ndarray, mask_u8: np.ndarray,
                  radius: int = 7) -> np.ndarray:
    """
    Fill crack pixels in the depth map by inpainting from surrounding surface.
    Uses INPAINT_TELEA (fast marching method — propagates from crack boundary
    inward, so filled values smoothly continue the surrounding surface).
    Returns float32 depth, same shape as input.
    """
    # cv2.inpaint needs 8-bit or 16-bit single-channel source
    d_min, d_max = depth_f32.min(), depth_f32.max()
    d_norm  = ((depth_f32 - d_min) / (d_max - d_min + 1e-6) * 65535).astype(np.uint16)
    filled  = cv2.inpaint(d_norm, mask_u8, radius, cv2.INPAINT_TELEA)
    # Back to original scale
    return filled.astype(np.float32) / 65535.0 * (d_max - d_min) + d_min


def inpaint_color(img_rgb: np.ndarray, mask_u8: np.ndarray,
                  radius: int = 7) -> np.ndarray:
    """
    Fill crack pixels in the RGB image using neighbouring surface texture.
    Gives the repaired mesh the visual appearance of the surrounding wall.
    """
    return cv2.inpaint(img_rgb, mask_u8, radius, cv2.INPAINT_TELEA)


def fill_crack_in_mesh(mesh_o3d,
                       filled_depth_f32: np.ndarray,
                       filled_img_rgb:   np.ndarray,
                       crack_mask_u8:    np.ndarray,
                       fx: float, fy: float,
                       cx: float, cy: float,
                       H: int, W: int) -> None:
    """
    Modify mesh vertices IN-PLACE:
      - For every vertex whose back-projected pixel falls inside the crack mask,
        replace its Z coordinate with the inpainted (flat) depth value.
      - Replace its vertex color with the inpainted surface texture color.

    The back-projection undoes the flip transform that was applied to the pcd:
        pcd.transform([[1,0,0,0],[0,-1,0,0],[0,0,-1,0],[0,0,0,1]])
    So: X_cam =  vx,  Y_cam = -vy,  Z_cam = -vz
    Pixel:  u = fx*(X_cam/Z_cam) + cx
            v = fy*(Y_cam/Z_cam) + cy
    """
    verts  = np.asarray(mesh_o3d.vertices)       # Nx3, writable copy below
    colors = np.asarray(mesh_o3d.vertex_colors)  # Nx3 float [0,1]

    verts_new  = verts.copy()
    colors_new = colors.copy()

    vx, vy, vz = verts[:, 0], verts[:, 1], verts[:, 2]

    # Undo flip to get camera-space coords
    X_cam =  vx
    Y_cam = -vy
    Z_cam = -vz

    valid = Z_cam > 1e-4
    u = np.full(len(verts), -1, dtype=int)
    v = np.full(len(verts), -1, dtype=int)
    u[valid] = np.clip(np.round(fx * X_cam[valid] / Z_cam[valid] + cx),
                       0, W - 1).astype(int)
    v[valid] = np.clip(np.round(fy * Y_cam[valid] / Z_cam[valid] + cy),
                       0, H - 1).astype(int)

    # Which vertices map to crack pixels?
    in_crack = valid & (crack_mask_u8[np.where(valid, v, 0),
                                      np.where(valid, u, 0)] > 10)

    if not in_crack.any():
        print("⚠️  No vertices mapped to crack region — mask may be too sparse.")
        return

    # Replace Z: convert filled depth (float32 on 0-1000 uint16 scale) to
    # the camera-space Z the vertex should have had.
    # filled_depth_f32 is in the same scale as the original final_depth that
    # generated depth_uint (0 → d_min, 1000 → d_max in uint16 space).
    # The RGBD pipeline uses depth_scale=1000, so Z_cam = depth_uint / 1000.
    # filled_depth_f32 is already on that 0-1000 scale → Z_new = filled/1000.
    Z_new_cam = filled_depth_f32[v[in_crack], u[in_crack]] / 1000.0
    # Recompute X, Y in camera space at the new Z
    X_new_cam = (u[in_crack].astype(float) - cx) * Z_new_cam / fx
    Y_new_cam = (v[in_crack].astype(float) - cy) * Z_new_cam / fy
    # Re-apply flip to get mesh coords
    verts_new[in_crack, 0] =  X_new_cam
    verts_new[in_crack, 1] = -Y_new_cam
    verts_new[in_crack, 2] = -Z_new_cam

    # Replace color from inpainted texture
    fill_colors = filled_img_rgb[v[in_crack], u[in_crack]].astype(np.float32) / 255.0
    colors_new[in_crack] = fill_colors

    mesh_o3d.vertices      = o3d.utility.Vector3dVector(verts_new)
    mesh_o3d.vertex_colors = o3d.utility.Vector3dVector(colors_new)

    # Recompute normals so shading is correct over the levelled area
    mesh_o3d.compute_vertex_normals()
    print(f"✅ Crack fill: {int(in_crack.sum()):,} vertices levelled.")


print("✅ Crack-fill helpers ready.")


In [ ]:
# ── Cell 6: Main pipeline ─────────────────────────────────────────────────────

def reconstruct_crack(input_image, crack_intensity: float = 50):
    """
    Returns (cracked_glb, filled_glb, report_PIL).

    cracked_glb  — original reconstruction showing the crack depression
    filled_glb   — levelled reconstruction with crack filled flush
    report_PIL   — 8-panel analysis image (PIL.Image)
    """
    report_pil = None

    if input_image is None:
        return None, None, None

    try:
        # ── 0. Channel normalisation ───────────────────────────────────────
        img = input_image
        if img.ndim == 2:
            img = np.stack([img] * 3, axis=-1)
        elif img.shape[2] == 4:
            img = img[:, :, :3]
        img = img.astype(np.uint8)

        # ── 1. Crack mask + displacement ──────────────────────────────────
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        _, mask_raw = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY_INV)

        # Dilate mask slightly so the inpaint boundary is clean
        kernel   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        mask_dil = cv2.dilate(mask_raw, kernel, iterations=2)

        mask_blur = cv2.GaussianBlur(mask_raw, (15, 15), 0).astype(np.float32)
        disp_map  = (mask_blur / 255.0) * float(crack_intensity)

        # ── 2. Depth inference ────────────────────────────────────────────
        base_depth = np.array(
            depth_pipe(Image.fromarray(img))["depth"]
        ).astype(np.float32)

        # ── 3. Align sizes ─────────────────────────────────────────────────
        dh, dw = base_depth.shape
        if (img.shape[0], img.shape[1]) != (dh, dw):
            disp_map  = cv2.resize(disp_map,  (dw, dh), interpolation=cv2.INTER_LINEAR)
            mask_raw  = cv2.resize(mask_raw,  (dw, dh), interpolation=cv2.INTER_NEAREST)
            mask_dil  = cv2.resize(mask_dil,  (dw, dh), interpolation=cv2.INTER_NEAREST)
            img       = cv2.resize(img,       (dw, dh), interpolation=cv2.INTER_LINEAR)

        # ── 4. Cracked depth (normalised to uint16 for RGBD) ───────────────
        final_depth  = base_depth + disp_map
        d_min, d_max = final_depth.min(), final_depth.max()
        depth_norm   = (final_depth - d_min) / (d_max - d_min + 1e-6)
        depth_uint   = (depth_norm * 1000).astype(np.uint16)  # 0-1000 scale

        # ── 5. Inpaint depth + color for the FILLED reconstruction ────────
        # filled_depth_f32 is on the same 0-1000 scale as depth_uint
        filled_depth_f32 = inpaint_depth(
            depth_uint.astype(np.float32), mask_dil, radius=9
        )
        filled_img_rgb = inpaint_color(img, mask_dil, radius=9)

        # Normalised versions for the report panels
        def norm01(arr):
            mn, mx = arr.min(), arr.max()
            return (arr - mn) / (mx - mn + 1e-6)

        # ── 6. Build analysis report ──────────────────────────────────────
        report_arr = build_report(
            original_rgb     = img,
            base_depth       = base_depth,
            crack_mask_raw   = mask_raw,
            displacement_map = disp_map,
            final_depth      = final_depth,
            filled_depth_norm = norm01(filled_depth_f32),
            filled_img_rgb   = filled_img_rgb,
            crack_intensity  = crack_intensity,
        )
        report_pil = Image.fromarray(report_arr)
        print("✅ Report rendered.")

        # ── Shared RGBD → pcd helper ──────────────────────────────────────
        H, W   = depth_uint.shape
        fx = fy = float(max(H, W))
        cx, cy  = W / 2.0, H / 2.0
        intr    = o3d.camera.PinholeCameraIntrinsic(W, H, fx, fy, cx, cy)
        FLIP    = [[1,0,0,0],[0,-1,0,0],[0,0,-1,0],[0,0,0,1]]

        def depth_to_pcd(depth_u16, color_rgb):
            c_o3d = o3d.geometry.Image(np.ascontiguousarray(color_rgb))
            d_o3d = o3d.geometry.Image(np.ascontiguousarray(depth_u16))
            rgbd  = o3d.geometry.RGBDImage.create_from_color_and_depth(
                c_o3d, d_o3d,
                depth_scale=1000.0, depth_trunc=3.0,
                convert_rgb_to_intensity=False)
            pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intr)
            pcd.transform(FLIP)
            pcd = pcd.voxel_down_sample(voxel_size=0.005)
            if len(pcd.points) < 500:
                return None
            pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
            pcd.estimate_normals(
                search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.02, max_nn=30))
            pcd.orient_normals_towards_camera_location([0.0, 0.0, 0.0])
            return pcd

        def pcd_to_mesh(pcd):
            mesh, _ = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
                pcd, depth=8)
            mesh = mesh.crop(pcd.get_axis_aligned_bounding_box())
            pc_pts = np.asarray(pcd.points)
            pc_col = np.asarray(pcd.colors)
            _, idx = cKDTree(pc_pts).query(np.asarray(mesh.vertices))
            mesh.vertex_colors = o3d.utility.Vector3dVector(pc_col[idx])
            return mesh

        tmp = tempfile.gettempdir()

        # ── 7. CRACKED mesh ────────────────────────────────────────────────
        print("Building cracked mesh…")
        pcd_crack = depth_to_pcd(depth_uint, img)
        if pcd_crack is None:
            print("⚠️  Too few points — skipping 3D meshes.")
            return None, None, report_pil

        mesh_crack = pcd_to_mesh(pcd_crack)
        glb_crack  = os.path.join(tmp, "crack_original.glb")
        _to_glb_with_colors(mesh_crack, glb_crack)
        print(f"✅ Cracked mesh → {len(mesh_crack.triangles):,} tris")

        # ── 8. FILLED mesh ────────────────────────────────────────────────
        # Strategy: build mesh from the inpainted depth + color so the point
        # cloud itself is already levelled, then additionally snap any
        # remaining crack-region vertices to the inpainted surface.
        print("Building filled (levelled) mesh…")
        filled_uint = np.clip(filled_depth_f32, 0, 1000).astype(np.uint16)
        pcd_fill   = depth_to_pcd(filled_uint, filled_img_rgb)
        if pcd_fill is None:
            print("⚠️  Filled point cloud too sparse — returning cracked mesh only.")
            return glb_crack, None, report_pil

        mesh_fill = pcd_to_mesh(pcd_fill)

        # Fine-grained vertex correction: project each vertex back to 2D,
        # and for any that still land in the crack mask, snap Z to the
        # inpainted depth and color to the inpainted texture.
        fill_crack_in_mesh(
            mesh_fill,
            filled_depth_f32 = filled_depth_f32,
            filled_img_rgb   = filled_img_rgb,
            crack_mask_u8    = mask_dil,
            fx=fx, fy=fy, cx=cx, cy=cy, H=H, W=W,
        )

        glb_fill = os.path.join(tmp, "crack_filled.glb")
        _to_glb_with_colors(mesh_fill, glb_fill)
        print(f"✅ Filled mesh → {len(mesh_fill.triangles):,} tris")

        return glb_crack, glb_fill, report_pil

    except Exception:
        traceback.print_exc()
        return None, None, report_pil


print("✅ reconstruct_crack() defined.")


In [ ]:
# ── Cell 7: Gradio UI ─────────────────────────────────────────────────────────

with gr.Blocks(title="Crack 3D Reconstruction", theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        "## 🪨 Crack 3D Reconstruction + Repair\n"
        "Upload a crack photo → **original cracked mesh**, "
        "**levelled/filled mesh**, and an **8-panel analysis report**."
    )

    with gr.Row():
        with gr.Column(scale=1, min_width=300):
            input_img = gr.Image(type="numpy", label="📷 Crack Image")
            intensity = gr.Slider(
                minimum=10, maximum=150, value=50, step=5,
                label="Crack Displacement Strength",
                info="10–40 = hairline  |  50–80 = moderate  |  90–150 = deep structural")
            run_btn = gr.Button("⚙️ Reconstruct + Repair", variant="primary")
            gr.Markdown(
                "_The report appears first. "
                "Both 3D models are generated with the same Poisson pipeline — "
                "the filled model uses inpainted depth and texture so the crack "
                "region is flush with the surrounding surface._"
            )

    with gr.Tabs():

        with gr.Tab("🪨 Cracked Mesh"):
            out_crack = gr.Model3D(
                label="Original — crack depression visible",
                clear_color=[0.12, 0.12, 0.12, 1.0],
            )
            gr.Markdown("_Drag to rotate · Scroll to zoom · Right-drag to pan_")

        with gr.Tab("🔧 Filled / Levelled Mesh"):
            out_fill = gr.Model3D(
                label="Repaired — crack filled flush with surrounding surface",
                clear_color=[0.12, 0.12, 0.12, 1.0],
            )
            gr.Markdown(
                "_Crack region depth is replaced by bilinear inpainting from "
                "the surrounding surface. Vertex colors are filled from neighbouring "
                "wall texture. Normals are recomputed for correct shading._"
            )

        with gr.Tab("📊 Analysis Report"):
            out_report = gr.Image(
                label="8-Panel Crack Analysis",
                show_download_button=True,
            )
            gr.Markdown(
                "**Panels (left→right, top→bottom):**\n"
                "Original · AI depth · Crack mask · Displacement heatmap\n"
                "Depth difference Δ · Severity overlay · Repaired surface (2D) · Levelled depth map\n\n"
                "Stats appear beneath the severity panel."
            )

    run_btn.click(
        fn=reconstruct_crack,
        inputs=[input_img, intensity],
        outputs=[out_crack, out_fill, out_report],
    )

demo.launch(share=True)
